# **Data Reduction**

This project uses 20GB of EEG time-series data to predict Schizophrenia from basic sensory task. To properly digest the data for Machine learning task, data need to be aggregated. 

Each of the 81 participants had 3 sensory tasks. With each condition, 100 trials were recorded for 3,072 samples taken per milliseconds. 

The goal of this notebook is to aggregate the samples to create one record representative of each trial

Dataset: 
[EEG data from basic sensory task in Schizophrenia](https://www.kaggle.com/datasets/broach/button-tone-sz/data)

### **Downloading files from Kaggle**

Files cannot be shared through GitHub nor Google Drive because of their size, so they have to be downloaded from original source: Kaggle.

In [ ]:
from pathlib import Path
from eeg_schizophrenia.download_data import download_dataset_csvs

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

summary = download_dataset_csvs(
    output_dir=repo_root / "data" / "raw",
    progress=True,
)

print(f"downloaded={len(summary.downloaded_paths)}, skipped={len(summary.skipped_paths)}")

### **Creating a Spark Session**

This Spark session is similar to the session created in the labs, with a few changes to avoid memory overload.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

import os
import sys

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

### **Loading the CSV Files**

The data for each participant is in a folder containing one CSV files. The column names for these data is in a separate file.

In [ ]:
# Confirming how many files we have

base_path = "data/raw"
data_paths = list(Path(base_path).rglob("*/*.csv"))
print(f"Found {len(data_paths)} files. " \
      f"From `{data_paths[0]}` to `{data_paths[-1]}`")

In [ ]:
# Reading column names

column_names = spark.read.csv(r".\data\raw\columnLabels.csv",
                              header=True).columns
print("List of column labels:")
for i, column in enumerate(column_names, start=1):
    print(f"  {i}. {column}" )

In [ ]:
# Creating a Spark DataFrame from all the data 

# df = spark.read.csv(r".small\raw\*\*.csv", 
df = spark.read.csv(r"data\raw\*\*.csv", 
                    inferSchema=True, # infer data types
                    header=False)
df = df.toDF(*column_names) 

row_count = df.count()
print(f"Read {row_count:,} records")

df.show(5)

### **Representing Time Series Data**

Each trial is its own time series data, they were aggregated using Symbolic Aggregate approXimation (SAX). SAX simplifies time series data by splitting it into equal segments—in this case, every 0.1 second—averaging each segment, and then converting those averages into symbols (letters). The result is a string with 30 letters that represents how the EEG data changes per trial.

In [ ]:
# Normalizing the DataFrame

from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array

no_normalize = ["subject", "trial", "condition", "sample"]
normalize_cols = [c for c in df.columns if c not in no_normalize]

assembler = VectorAssembler(inputCols=normalize_cols, outputCol="features")
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)
df_vec = assembler.transform(df)
df_vec = scaler.fit(df_vec).transform(df_vec)

# The scaled values were put into its own column (that really confused me.) 
# so, I unfolded that column and replaced the original features with the 
# scaled features.

normalized_df = df_vec.select(
    *no_normalize,
    *[vector_to_array("scaledFeatures")[i].alias(c)for i, c in enumerate(normalize_cols)]
)

normalized_df.show(5)

In [ ]:
# Creating SAX strings
# Dividing into 30 bins, getting the average of each bin, mapping 
# each value to a letter, then combining the letters into one string

normalized_df.createOrReplaceTempView("normalized_table")

sax_exprs = [
    f"""
    CASE 
        WHEN AVG({c}) < -0.67 THEN 'a'
        WHEN AVG({c}) < 0 THEN 'b'
        WHEN AVG({c}) < 0.67 THEN 'c'
        ELSE 'd'
    END AS {c}_sax
    """
    for c in normalize_cols
]

final_exprs = [
    f"""
    CONCAT_WS(
        '',
        TRANSFORM(
            SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT('g', group_id, 'v', {c}_sax))),
            x -> x.v
        )
    ) AS {c}_sax
    """
    for c in normalize_cols
]

# I asked GPT how to make my manual SAX code faster and 
# it said turn "PAA -> SAX -> Concat" into one SQL query  
# My old code that uses df.groupBy is in commit d5312bc1e5f31688c806054d731e824a8f844779

query = f"""
WITH base AS (
    SELECT
        subject,
        trial,
        condition,
        sample,
        NTILE(30) OVER ( 
            PARTITION BY subject, trial, condition
            ORDER BY sample
        ) - 1 AS group_id,
        {", ".join(normalize_cols)}
    FROM normalized_table
),

paa_sax AS (
    SELECT
        subject,
        trial,
        condition,
        group_id,
        {", ".join(sax_exprs)}
    FROM base
    GROUP BY subject, trial, condition, group_id
)

SELECT
    subject,
    trial,
    condition,
    {", ".join(final_exprs)}
FROM paa_sax
GROUP BY subject, trial, condition
"""

sax_df = spark.sql(query)
sax_df.show(5)

In [ ]:

# I asked GPT how to make my manual SAX code faster and 
# it said turn "PAA -> SAX -> Concat" into one SQL query  
# My old code that uses df.groupBy is in commit d5312bc1e5f31688c806054d731e824a8f844779

query = f"""
WITH base AS (
    SELECT
        subject,
        trial,
        condition,
        sample,
        NTILE(30) OVER ( 
            PARTITION BY subject, trial, condition
            ORDER BY sample
        ) - 1 AS group_id,
        {", ".join(normalize_cols)}
    FROM normalized_table
),

paa_sax AS (
    SELECT
        subject,
        trial,
        condition,
        group_id,
        {", ".join(sax_exprs)}
    FROM base
    GROUP BY subject, trial, condition, group_id
)

SELECT
    subject,
    trial,
    condition,
    {", ".join(final_exprs)}
FROM paa_sax
GROUP BY subject, trial, condition
"""

sax_df = spark.sql(query)
sax_df.show(5)

### **Aggregating Data**

Back to working with unnormalized data, some features maybe be missed with just SAX so I wanted to get the average, maximum, minimum, and standard deviation of each feature for each trial to help with the prediction.

In here I wanted to add `mode` as well but since mode is a heavy process, it caused memory errors. I also tried adding `q1`, `median`, and `q3` but those processes also involve sorting which was also too heavy for our local machines

In [ ]:
from pyspark.sql.functions import avg, max, min, stddev

group_cols = ["subject", "trial", "condition"]
other_cols = [col_name for col_name in df.columns if col_name not in group_cols]
other_cols.remove("sample")

grp_functions = [(avg, "avg"),(max, "max"),(min, "min"),(stddev, "stddev")]

agg_df = df.groupBy(group_cols).agg(
                *[f(c).alias(f"{c}_{name}") for c in other_cols for f, name in grp_functions]
            ).orderBy(group_cols)
agg_df.show(15)

At this point we combined the two DataFrames and ran a baseline logistic regression with TFIDF for the SAX columns. To avoid leakage, we split the data by subject, so trials from the same person did not appear in both training and testing. However, the model performed poorly: holdout accuracy was only 0.4802, balanced accuracy was 0.4326, F1 for the positive class was 0.2599, and ROC-AUC was 0.3504. Cross-validation results were similarly weak, with ROC-AUC around 0.4825, which is close to flipping a coin.

Before looking into bigger models, we want to first try to reduce the data better.

### **Extracting Event-Related Potential (ERP) Components**

The dataset mentions extracting features from specific time windows called Event-Related Potential (ERP) components:

* Baseline (B0, B1) before stimulus
    * B0 is the main baseline between -100 to 0ms (inclusive) 
    * B1 is the alternative, earlier baseline between -600 to -500ms (inclusive) 
* N100 (~100 ms) early response between 80-100ms (inclusive)
* P200 (~200 ms) later response 150-190ms (inclusive)

In [ ]:
# Reading the time map csv
time_sample_map = spark.read.csv(r".\data\raw\time.csv",
                              inferSchema=True, # infer data types
                              header=True)
time_sample_map = time_sample_map.toDF(*[c.strip() for c in time_sample_map.columns]) #there a space on column
time_sample_map.printSchema()
time_sample_map.show(5)

In [ ]:
# Adding ERP labels to time map

from pyspark.sql.functions import when, col

time_sample_map = time_sample_map.withColumn(
    "erp",
    when((col('time_ms')  >= -600) & (col('time_ms') <= -500), "B1")   #100ms
    .when((col('time_ms') >= -100) & (col('time_ms') <= 0),    "B0")   #100ms
    .when((col('time_ms') >= 80)   & (col('time_ms') <= 100),  "N100") #20ms
    .when((col('time_ms') >= 150)  & (col('time_ms') <= 190),  "P200") #40ms
)
time_sample_map = time_sample_map.filter(col('erp').isNotNull())
time_sample_map.show(5)


In [ ]:
# Applying aggregate functions to ERP components

erp_agg_df = df.join(time_sample_map.select("sample", 'erp'), on="sample", how='inner')

agg_exprs = []

erp_col = "erp"
erp_cols = [row["erp"] for row in time_sample_map.select("erp").distinct().collect()]

group_cols = ["subject", "trial", "condition"]
other_cols = [col_name for col_name in df.columns if col_name not in group_cols]
other_cols.remove("sample")

from pyspark.sql.functions import avg, min, max, stddev
grp_functions = [(avg, "avg"),(max, "max"),(min, "min"),(stddev, "stddev")]

erp_agg_df = erp_agg_df.groupBy(group_cols).agg(
                *[  f(when(col(erp_col)== erp, col(c))).alias(f"{c}_{erp}_{agg_name}") 
                  for c in other_cols 
                  for f, agg_name in grp_functions 
                  for erp in erp_cols ]
            ).orderBy(*group_cols)

erp_agg_df.show(5)

In [ ]:
# Adding SAX of the most critical 300ms instead of the whole 3000ms

# Dividing into 15 bins of 20ms each, getting the average of each bin, mapping 
# each value to a letter, then combining the letters into one string

b0_start, p200_end = time_sample_map.agg(
    min(when(col("erp") == "B0", col("sample"))),  # This is the start of the ERP SAX
    max(when(col("erp") == "P200", col("sample"))) # This is the end of the ERP SAX
).first()

sax_exprs = [
    f"""
    CASE 
        WHEN AVG({c}) < -0.67 THEN 'a'
        WHEN AVG({c}) < 0 THEN 'b'
        WHEN AVG({c}) < 0.67 THEN 'c'
        ELSE 'd'
    END AS {c}_erp_sax
    """
    for c in normalize_cols
]

final_exprs = [
    f"""
    CONCAT_WS(
        '',
        TRANSFORM(
            SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT('g', group_id, 'v', {c}_erp_sax))),
            x -> x.v
        )
    ) AS {c}_erp_sax
    """
    for c in normalize_cols
]


query = f"""
WITH base AS (
    SELECT
        subject,
        trial,
        condition,
        sample,
        NTILE(15) OVER ( 
            PARTITION BY subject, trial, condition
            ORDER BY sample
        ) - 1 AS group_id,
        {", ".join(normalize_cols)}
    FROM normalized_table
    WHERE sample BETWEEN {b0_start} AND {p200_end}
),

paa_sax AS (
    SELECT
        subject,
        trial,
        condition,
        group_id,
        {", ".join(sax_exprs)}
    FROM base
    GROUP BY subject, trial, condition, group_id
)

SELECT
    subject,
    trial,
    condition,
    {", ".join(final_exprs)}
FROM paa_sax
GROUP BY subject, trial, condition
"""

erp_sax_df = spark.sql(query)
erp_sax_df.show(5)

### **Saving the DataFrame**

The dataframe will be used for EDA, preprocessing, model training, and prediction. Saved in `data/processed/all_csv_agg`

In [ ]:
# Combining the processed DataFrames
all_df = agg_df \
    .join(sax_df, on=group_cols, how="left") \
    .join(erp_agg_df, on=group_cols, how="left") \
    .join(erp_sax_df, on=group_cols, how="left")

# Ordering them for readbility
agg_order = ["avg", "max", "min", "stddev"]
erp_order = ["B0", "B1", "N100", "P200"]

col_order = group_cols.copy()
for c in other_cols:
    col_order.extend([f"{c}_{agg}" for agg in agg_order])
    col_order.append(f"{c}_sax")
    col_order.append(f"{c}_erp_sax")
    col_order.extend(f"{c}_{erp}_{agg}" for erp in erp_order for agg in agg_order)
col_order = [c for c in col_order if c in all_df.columns]


all_df = all_df.select(*col_order).orderBy(*group_cols)
all_df.show(5)

In [ ]:
# Downloading the reduced csv
all_df.coalesce(1).write.mode("overwrite").option("header", True).csv("data/processed/all_csv_agg")

### **Stop Spark Session**

In [ ]:
spark.stop()